# Projet RAG — Pipeline complet avec comparaison des méthodes

**PFA — Stage d'été — 4ème Année IIR — Encadrante : Dr. EL MKHALET MOUNA**

Ce notebook est la **source de vérité** du projet : il implémente et exécute de bout en bout toutes les étapes du pipeline RAG sur un corpus réel (thèse académique de 191 pages), puis compare les méthodes de chaque étape :

1. **Cleaning du Data** (PDF brut → corpus ~100 pages, conforme au guide de l'encadrante)
2. **Concepts RAG** : principe, utilité, cas d'usage, avantages, étapes clés
3. **Chunking** : comparaison de **7 méthodes** (8 métriques)
4. **Embedding** : comparaison de **7 méthodes** (8 métriques)
5. **Retrieval** : comparaison de **6 méthodes** (précision@5, rappel@5, MRR, MAP@5, nDCG@5, cosine sim., latence)
6. **Génération (LLM)** : démo Ollama + repli Hugging Face
7. **Architecture du projet** : schéma généré + export
8. **Exports JSON** : consommés par l'application web (FastAPI + React)

**Prérequis d'exécution** :
- Environnement : `pip install -r requirements.txt` (Python 3.13 recommandé)
- Corpus brut : `thèse_pdf.pdf` dans le même dossier que ce notebook
- Optionnel (méthode 7 d'embedding + démo génération) : Ollama lancé (`ollama serve`) avec les modèles `nomic-embed-text` et `llama3.2:1b` (`ollama pull <modele>`)
- Première exécution avec internet : téléchargement des modèles pré-entraînés (GloVe ~130 Mo, BERT ~440 Mo, MiniLM ~90 Mo, cross-encoder ~440 Mo)

--- 

# PARTIE 0 — Concepts fondamentaux du RAG (Retrieval-Augmented Generation)

> *Cellule académique demandée par le sujet : comprendre avant d'implémenter.*

## Principe de fonctionnement

Un LLM (Large Language Model) génère des réponses à partir de ses connaissances **figées à sa date d'entraînement**. Il ignore tout de vos documents internes (rapports, thèses, bases de connaissances) et peut « halluciner » (inventer des faits plausibles). Le **RAG** corrige ces deux limites en deux temps :

```
                    ┌────────────────────────  HORS-LIGNE (indexation)  ────────────────────────┐
 Document(s) ──► CLEANING ──► CHUNKING (découpage) ──► EMBEDDING (vectorisation) ──► INDEX vectoriel
                    └────────────────────────────────────────────────────────────────────────┘
                    ┌────────────────────────── EN-LIGNE (requête)  ──────────────────────────┐
 Question ──► EMBEDDING de la question ──► RETRIEVAL (recherche des top-k passages) ──►
            ──► (optionnel : RE-RANKING) ──► PROMPT = question + passages ──► LLM ──► RÉPONSE + SOURCES
                    └────────────────────────────────────────────────────────────────────────┘
```

- **Retrieval** : avant de répondre, le système **cherche** les passages les plus pertinents du corpus (par similarité sémantique, mots-clés, ou les deux).
- **Augmented** : les passages récupérés sont **injectés dans le prompt** du LLM.
- **Generation** : le LLM rédige sa réponse **uniquement à partir de ces passages** (approche « à livre ouvert »), ce qui réduit les hallucinations et permet de **citer les sources**.

## Utilité et cas d'usage

| Cas d'usage | Exemple concret |
|---|---|
| Assistance documentaire | chatbot interne branché sur la documentation d'entreprise |
| Veille scientifique | interroger un corpus de thèses/articles (notre cas) |
| Support client | FAQ augmentée des tickets historiques |
| Conformité / juridique | retrouver les clauses exactes dans des contrats |
| Éducation | tutor interactif sur les polycopiés d'un cours |

## Avantages du RAG

1. **Fraîcheur** : le corpus change, pas besoin de ré-entraîner le modèle — il suffit de ré-indexer.
2. **Traçabilité** : chaque réponse cite les passages sources (vérifiable par l'humain).
3. **Réduction des hallucinations** : le LLM est contraint de s'appuyer sur des extraits réels.
4. **Coût** : bien moins cher qu'un fine-tuning, et compatible avec des LLM locaux (Ollama) → données privées.
5. **Contrôle d'accès** : on peut filtrer les documents indexés par utilisateur/rôle.

## Les 4 étapes clés (celles comparées dans ce projet)

1. **Découpage (chunking)** : découper le corpus en morceaux (« chunks ») assez courts pour porter une idée unique. Trop gros → l'embedding dilue le sens ; trop petits → le contexte est perdu. **7 méthodes comparées en Partie 2.**
2. **Vectorisation (embedding)** : transformer chaque chunk en vecteur numérique qui capture son **sens** (deux textes proches sémantiquement ont des vecteurs proches). **7 méthodes comparées en Partie 3.**
3. **Recherche d'information (retrieval)** : retrouver les top-k chunks les plus pertinents pour une question (exact vs approximatif, lexical vs sémantique, 1 ou 2 étages). **6 méthodes comparées en Partie 4.**
4. **Traitement des requêtes (query)** : construire le prompt final (question + passages) et générer la réponse avec citations — **démontré en Partie 5 avec Ollama / Hugging Face**.

---



## Principales bibliothèques RAG

*Demandée par le jury (panneau 1) : connaître les principales bibliothèques du domaine RAG et leur rôle dans ce projet.*

| Bibliothèque | Rôle dans un pipeline RAG | Où dans ce projet |
|---|---|---|
| **pypdf** | extraction du texte des PDF, page par page | Partie 1 (cleaning) |
| **LangChain** — `langchain-text-splitters`, `langchain-community`, `langchain-huggingface` | découpage du texte (splitters), connecteurs d'embeddings, vectorstores (FAISS) | TP1 + Partie 2 (chunking) |
| **sentence-transformers** | modèles d'embedding pré-entraînés (MiniLM, SBERT, multi-qa) + cross-encoder (re-ranking) | Parties 2, 3 et 4 + moteur live de l'application |
| **gensim** | Word2Vec, FastText et Doc2Vec entraînés from scratch (baselines à comparer) | Partie 3 (embedding) |
| **faiss-cpu** | index vectoriel ANN (HNSW) : recherche approximative ultra-rapide | TP1 + Partie 4 (retrieval) |
| **rank_bm25** | score BM25 : recherche par mots-clés (lexicale) | Partie 4 (retrieval) |
| **scikit-learn** | similarité cosinus, métriques | toutes les comparaisons |
| **Ollama** | LLM local : génération des réponses sourcées + embeddings (nomic-embed-text) | Partie 5 + application |
| **FastAPI + Uvicorn** | backend : API REST (résultats, meilleures méthodes, requête live) | application web (TP3) |
| **React + Vite + Recharts** | frontend : interface interactive (graphiques, podium, console de question) | application web (TP3) |

---



# PARTIE 1 — Cleaning du Data : PDF brut → corpus ~100 pages

**Consignes de l'encadrante (Catégorie 2 — sujet entreprise + volet académique)** :
- I. Cleaning : supprimer la partie remerciements, le sommaire, la liste des figures et la liste des tableaux.
- II. Traitement : après le cleaning, réduire le dataset à ~100 pages.

**Méthode** (automatisée, reproductible) :
1. Extraction texte page par page (`pypdf`).
2. Détection des pages de front-matter (remerciements, sommaire, listes) par motifs d'en-têtes + heuristique de « pages de liste » (lignes se terminant par des numéros de page).
2bis. Détection de la couverture d'archive (page « HAL Id: … » déposée sur archives-ouvertes.fr) parmi les 3 premières pages — boilerplate administratif en anglais, sans valeur pour le RAG.
3. Détection du end-matter (bibliographie, annexes) — peu utile pour un RAG — à partir de la moitié du document.
4. Troncature au budget de 100 pages (on garde le début : introduction générale et corps du rapport).
5. Statistiques avant/après + visualisation.


In [ ]:
import re
import numpy as np
import pandas as pd
from pypdf import PdfReader

# ---------------- Détection des sections non pertinentes ----------------
SECTION_HEADERS = [
    r"REMERCIEMENTS?", r"SOMMAIRE", r"TABLE\s+DES\s+MATI[EÈ]RES",
    r"LISTE\s+DES\s+FIGURES", r"LISTE\s+DES\s+TABLEAUX", r"TABLE\s+DES\s+FIGURES",
    r"TABLE\s+DES\s+TABLEAUX", r"TABLE\s+DES\s+ILLUSTRATIONS",
    r"LISTE\s+DES\s+ILLUSTRATIONS", r"LISTE\s+DES\s+ABR[EÉ]VIATIONS",
    r"R[EÉ]SUM[EÉ]\s*$", r"ABSTRACT\s*$",
]
HEADER_RE = re.compile("|".join(SECTION_HEADERS), re.IGNORECASE)
# ligne de sommaire : se termine par des points de suite + numéro de page
TOC_LINE_RE = re.compile(r"(\.{2,}\s*\d{1,4}\s*$)|(\s\d{1,4}\s*$)")
END_MATTER_RE = re.compile(r"^\s*(BIBLIOGRAPHIE|R[EÉ]F[EÉ]RENCES|ANNEXES?|TABLE\s+DES\s+MATI[EÈ]RES\s+D[EÉ]TAILL[EÉ]E)\s*$", re.IGNORECASE)

HAL_COVER_RE = re.compile(r"HAL\s+Id\s*:", re.IGNORECASE)   # couverture d'archive (dépôt HAL) : boilerplate hors thèse

def extract_pages(path):
    reader = PdfReader(path)
    return [p.extract_text() or "" for p in reader.pages]

def looks_like_list_page(page_text, min_ratio=0.35):
    lines = [l for l in page_text.split("\n") if l.strip()]
    if not lines:
        return False
    hits = sum(1 for l in lines if TOC_LINE_RE.search(l.strip()))
    return (hits / len(lines)) >= min_ratio

def find_cover_pages(pages, max_first=3):
    # Couverture d'archive (ex. page HAL déposée sur archives-ouvertes.fr) :
    # boilerplate administratif en anglais, sans valeur pour le RAG.
    cover = set()
    for i, text in enumerate(pages[:max_first]):
        if HAL_COVER_RE.search(text[:400]):
            cover.add(i)
    return cover

def find_removable_pages(pages):
    removable, in_section = set(), False
    for i, text in enumerate(pages):
        head = "\n".join(text.split("\n")[:4])
        if HEADER_RE.search(head):
            in_section = True
            removable.add(i)
            continue
        if in_section:
            if looks_like_list_page(text):
                removable.add(i)
            else:
                in_section = False
    return removable

def find_end_matter_start(pages):
    for i, text in enumerate(pages):
        if i < int(len(pages) * 0.5):   # le end-matter ne commence jamais avant la moitié
            continue
        first_lines = [l.strip() for l in text.split("\n")[:5] if l.strip()][:2]
        if any(END_MATTER_RE.match(l) for l in first_lines):
            return i
    return None

def clean_and_reduce(path, page_budget=100):
    pages = extract_pages(path)
    total = len(pages)
    removable = find_removable_pages(pages)
    cover = find_cover_pages(pages)
    removable |= cover
    end_start = find_end_matter_start(pages)
    if end_start is not None:
        removable.update(range(end_start, total))
    kept_idx = [i for i in range(total) if i not in removable]
    after_cleaning = len(kept_idx)
    truncated = False
    if len(kept_idx) > page_budget:
        kept_idx = kept_idx[:page_budget]
        truncated = True
    cleaned_text = "\n\n".join(pages[i] for i in kept_idx)
    return {
        "path": path, "total_pages": total,
        "cover_pages_removed": sorted(i + 1 for i in cover),
        "front_matter_removed": sorted(i + 1 for i in removable if end_start is None or i < end_start),
        "end_matter_start_page": (end_start + 1) if end_start is not None else None,
        "pages_after_cleaning": after_cleaning,
        "final_page_count": len(kept_idx),
        "truncated_to_budget": truncated,
        "text": cleaned_text,
    }

print("Fonctions de cleaning prêtes.")


In [ ]:
PDF_PATH = "thèse_pdf.pdf"
PAGE_BUDGET = 100

info = clean_and_reduce(PDF_PATH, page_budget=PAGE_BUDGET)

print(f"Fichier                     : {info['path']}")
print(f"Pages totales (PDF brut)    : {info['total_pages']}")
print(f"Couverture archive retirée  : {info['cover_pages_removed']}")
print(f"Pages front-matter retirées : {info['front_matter_removed']}")
print(f"Bibliographie/annexes dès   : p.{info['end_matter_start_page']}")
print(f"Pages après cleaning        : {info['pages_after_cleaning']}")
print(f"Pages finales (budget {PAGE_BUDGET})   : {info['final_page_count']} (tronqué : {info['truncated_to_budget']})")


In [ ]:
import os

corpus_text = info["text"]
os.makedirs("data", exist_ok=True)
with open("data/CORPUS_RAG_reduit_100pages.txt", "w", encoding="utf-8") as f:
    f.write(corpus_text)

# Statistiques avant / après
raw_pages = extract_pages(PDF_PATH)
raw_text = "\n".join(raw_pages)
stats = {
    "": ["PDF brut", "Corpus nettoyé"],
    "Pages": [info["total_pages"], info["final_page_count"]],
    "Mots": [len(raw_text.split()), len(corpus_text.split())],
    "Caractères": [len(raw_text), len(corpus_text)],
}
stats_df = pd.DataFrame(stats).set_index("")
print(stats_df.to_string())
print(f"\nCorpus sauvegardé -> data/CORPUS_RAG_reduit_100pages.txt ({len(corpus_text)} caractères)")


In [ ]:
# Visualisation du cleaning (pages conservées vs retirées)
import matplotlib.pyplot as plt

removed_front = len(info["front_matter_removed"])
end_start = info["end_matter_start_page"]
removed_end = (info["total_pages"] - end_start + 1) if end_start else 0
kept = info["final_page_count"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(["Retiré\n(front-matter)", "Retiré\n(biblio/annexes)", "Conservé\n(corpus final)"],
              [removed_front, removed_end, kept],
              color=["#e07a5f", "#f2cc8f", "#81b29a"])
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1, str(int(b.get_height())),
            ha="center", fontweight="bold")
ax.set_ylabel("Nombre de pages")
ax.set_title(f"Cleaning du PDF : {info['total_pages']} pages -> {kept} pages (~{PAGE_BUDGET} visées)")
plt.tight_layout()
plt.savefig("docs/cleaning_pages.png", dpi=130)
plt.show()


In [ ]:
# Aperçu du début et de la fin du corpus final (contrôle qualité)
print("=== DÉBUT DU CORPUS (600 premiers caractères) ===")
print(corpus_text[:600])
print("\n=== FIN DU CORPUS (300 derniers caractères) ===")
print(corpus_text[-300:])


**Lecture du résultat** : le PDF brut (191 pages) est réduit à **100 pages** de contenu utile. Les pages retirées sont exactement celles interdites par les consignes (remerciements, sommaire, listes de figures/tableaux), la couverture d'archive HAL (boilerplate hors thèse) et le end-matter (bibliographie + annexes, inutiles pour un RAG). Le corpus commence donc directement par la page de titre de la thèse. Le fichier `data/CORPUS_RAG_reduit_100pages.txt` est le corpus officiel utilisé par **toutes** les comparaisons suivantes.

---


# PARTIE 1bis — TP1 : Pipeline RAG simplifié (démo sur un extrait du corpus)

Reprise du TP1 sur un **extrait réel du corpus** (20 premières phrases) pour bien fixer les idées avant les comparaisons :
chunking → embeddings → vector store FAISS → similarité cosinus → retrieval. Livrables TP1 inclus (matrice de similarité + heatmap).

---



In [ ]:
# Extrait du corpus pour la démo TP1
# (simple_sentence_split est défini ici pour l'autonomie de la démo)
import re as _re

def simple_sentence_split(t):
    sentences = _re.split(r"(?<=[.!?])\s+", t.strip())
    return [s.strip() for s in sentences if s.strip()]

demo_text = " ".join(simple_sentence_split(corpus_text)[:20])
print(f"Extrait démo : {len(demo_text)} caractères, {len(simple_sentence_split(demo_text))} phrases")
print(demo_text[:300], "...")


In [ ]:
# Etape TP1 : chunking (par phrase), embeddings, FAISS, matrice de similarité
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

splitter = CharacterTextSplitter(separator="\n", chunk_size=100, chunk_overlap=20)
demo_docs = [d.strip() for d in splitter.split_text(demo_text.replace(". ", ".\n")) if d.strip()]

demo_embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
demo_chunk_embs = np.array(demo_embeddings_model.embed_documents(demo_docs))
print(f"Chunks : {len(demo_docs)} | Dimension des vecteurs : {demo_chunk_embs.shape[1]}")

demo_vectorstore = FAISS.from_texts(demo_docs, demo_embeddings_model)
print("Vector store FAISS créé avec", demo_vectorstore.index.ntotal, "vecteurs.")


In [ ]:
# Livrable TP1 : matrice de similarité cosinus entre chunks + heatmap
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

demo_sim = cosine_similarity(demo_chunk_embs)
labels = [f"Chunk {i+1}" for i in range(len(demo_docs))]
demo_sim_df = pd.DataFrame(demo_sim, index=labels, columns=labels).round(3)

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(demo_sim_df, cmap="Blues", vmin=0, vmax=1, ax=ax)
ax.set_title("TP1 — Matrice de similarité cosinus entre chunks (extrait du corpus)")
plt.tight_layout()
plt.savefig("docs/tp1_similarity_matrix.png", dpi=130)
plt.show()


In [ ]:
# TP1 : retrieval d'une question sur l'extrait (FAISS + cosinus manuel)
demo_query = "Comment les zones humides réduisent-elles la pollution par les nitrates ?"

results = demo_vectorstore.similarity_search_with_score(demo_query, k=2)
print(f"Question : {demo_query}\n--- Résultats FAISS (distance L2, plus petit = plus proche) ---")
for rank, (doc, score) in enumerate(results, 1):
    print(f"#{rank} (distance = {score:.4f}) : {doc.page_content[:120]}")

q_emb = np.array(demo_embeddings_model.embed_query(demo_query)).reshape(1, -1)
sims = cosine_similarity(q_emb, demo_chunk_embs)[0]
best = int(np.argmax(sims))
print(f"\nChunk le plus pertinent (cosinus manuel) : Chunk {best+1} (similarité = {sims[best]:.4f})")
print(demo_docs[best][:200])


# PARTIE 2 — Comparaison des 7 méthodes de Chunking

Le corpus est fixé (celui du cleaning). On ne fait varier **que** la méthode de découpage : c'est la règle d'or de l'expérimentation.

## Les 7 méthodes implémentées

| # | Méthode | Principe | Points forts | Limites |
|---|---|---|---|---|
| 1 | **Taille fixe** | tranches brutes de N caractères | simple, rapide | coupe les mots et les idées |
| 2 | **Récursif** | essaie séparateurs hiérarchiques (paragraphe > phrase > mot) vers une taille cible | bon compromis universel | pas de notion de sens |
| 3 | **Par phrase** | regroupe des phrases complètes jusqu'à la taille cible | jamais de phrase coupée | ignore les transitions thématiques |
| 4 | **Par paragraphe** | un chunk = un paragraphe naturel | respecte la structure d'origine | sur du texte PDF, les paragraphes sont irréguliers |
| 5 | **Fenêtre glissante** | fenêtres de mots qui se chevauchent | redondance contrôlée, bon recall | duplique le contenu (coût d'index) |
| 6 | **Par tokens** | découpe selon les vrais tokens du LLM (tiktoken) | cohérent avec les limites des LLM | même aveugle au sens que (1) |
| 7 | **Sémantique** | regroupe les phrases consécutives tant que leur similarité d'embedding reste haute | chunks thématiquement cohérents | coûteux (N embeddings de phrases) |

> **Note d'implémentation** : la méthode sémantique nécessite un embedding de chaque phrase et des comparaisons de phrases consécutives ; pour garder un temps d'exécution raisonnable, elle est appliquée sur un **extrait contigu représentatif** du corpus (les 40 000 premiers caractères, ≈ 20 % du corpus). Les 6 autres méthodes traitent l'intégralité du corpus.


In [ ]:
import numpy as np
import pandas as pd
import re

np.random.seed(42)

# ---------- Les 7 fonctions de chunking ----------
def chunk_fixed_size(t, size=180):
    return [t[i:i+size] for i in range(0, len(t), size) if t[i:i+size].strip()]

def chunk_recursive(t, size=180, overlap=30):
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    return splitter.split_text(t)

def chunk_by_sentence(t, target_size=180):
    sentences = simple_sentence_split(t)
    chunks, current = [], ""
    for s in sentences:
        if len(current) + len(s) + 1 <= target_size or not current:
            current = (current + " " + s).strip()
        else:
            chunks.append(current)
            current = s
    if current:
        chunks.append(current)
    return chunks

def chunk_by_paragraph(t):
    paras = [p.strip() for p in t.split("\n\n") if p.strip()]
    return paras

def chunk_sliding_window(t, window_words=30, stride_words=15):
    words = t.split()
    chunks = []
    for i in range(0, len(words), stride_words):
        window = words[i:i+window_words]
        if not window:
            break
        chunks.append(" ".join(window))
        if i + window_words >= len(words):
            break
    return chunks

def chunk_by_tokens(t, chunk_size=40, chunk_overlap=8):
    from langchain_text_splitters import TokenTextSplitter
    splitter = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(t)

def chunk_semantic(t, embed_fn, similarity_threshold=0.55, max_chars=260, max_chars_processed=40_000):
    # appliqué sur un extrait contigu (voir note d'implémentation)
    extract = t[:max_chars_processed]
    sentences = simple_sentence_split(extract)
    if not sentences:
        return []
    embs = np.array(embed_fn(sentences))
    from sklearn.metrics.pairwise import cosine_similarity
    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        sim = cosine_similarity([embs[i-1]], [embs[i]])[0][0]
        current_len = sum(len(s) for s in current)
        if sim >= similarity_threshold and current_len < max_chars:
            current.append(sentences[i])
        else:
            chunks.append(" ".join(current))
            current = [sentences[i]]
    chunks.append(" ".join(current))
    return chunks

# Modèle d'embedding partagé (MiniLM) : sert au chunking sémantique ET aux métriques
from langchain_huggingface import HuggingFaceEmbeddings
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def embed_fn(texts):
    return embeddings_model.embed_documents(texts)

print("7 fonctions de chunking définies + modèle d'embedding chargé.")


In [ ]:
# ---------- Application des 7 méthodes sur le corpus ----------
methods = {
    "1. Taille fixe": chunk_fixed_size(corpus_text),
    "2. Récursif": chunk_recursive(corpus_text),
    "3. Par phrase": chunk_by_sentence(corpus_text),
    "4. Par paragraphe": chunk_by_paragraph(corpus_text),
    "5. Fenêtre glissante": chunk_sliding_window(corpus_text),
    "6. Par tokens": chunk_by_tokens(corpus_text),
    "7. Sémantique": chunk_semantic(corpus_text, embed_fn),
}

for name, chunks in methods.items():
    lengths = [len(c) for c in chunks]
    print(f"{name:22s} : {len(chunks):5d} chunks | longueur moy. {np.mean(lengths):6.1f} car. | min {min(lengths)} / max {max(lengths)}")


## Les 8 métriques d'évaluation

| Métrique | Objectif | Calcul |
|---|---|---|
| Nombre de chunks | fragmentation (coût d'index) | `len(chunks)` |
| Longueur moyenne | taille optimale | moyenne des caractères par chunk |
| Écart-type | homogénéité des tailles | écart-type des longueurs |
| Similarité intra | **cohérence** interne | cosinus moyen entre les phrases **à l'intérieur** de chaque chunk (↑ mieux) |
| Similarité inter | **redondance** | cosinus moyen **entre chunks différents** (↓ mieux) |
| Recall@2 | couverture | le bon chunk est-il dans le top-2 ? |
| Precision@2 | pertinence | proportion pertinente sur les 2 slots |
| F1-score | performance globale | moyenne harmonique de Precision@2 et Recall@2 |
| MRR (+bonus) | position du bon résultat | moyenne de 1/rang du bon chunk |

**Méthodologie Recall/Precision/F1/MRR** (interprétation raisonnée, à valider avec l'encadrante) : chaque phrase du corpus sert de **pseudo-question** ; le chunk « correct » est celui qui contient cette phrase. On embed la pseudo-question et tous les chunks, on classe par similarité cosinus et on mesure le rang du bon chunk.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def length_stats(chunks):
    lengths = [len(c) for c in chunks]
    return len(chunks), float(np.mean(lengths)), float(np.std(lengths))

def intra_inter_similarity(chunks, chunk_embs, embed_fn, max_sentences=3500):
    """intra : cosinus moyen entre phrases d'un même chunk ; inter : cosinus moyen entre chunks."""
    # --- intra : on rassemble toutes les phrases avec leur chunk propriétaire (échantillon capé) ---
    all_sents, owner = [], []
    for ci, c in enumerate(chunks):
        for s in simple_sentence_split(c):
            all_sents.append(s)
            owner.append(ci)
    if len(all_sents) > max_sentences:  # échantillonnage déterministe (stride)
        stride = int(np.ceil(len(all_sents) / max_sentences))
        all_sents = all_sents[::stride]
        owner = owner[::stride]
    sent_embs = np.array(embed_fn(all_sents)) if all_sents else np.zeros((0, 1))
    intra_scores = []
    by_chunk = {}
    for emb, ci in zip(sent_embs, owner):
        by_chunk.setdefault(ci, []).append(emb)
    for ci, embs in by_chunk.items():
        n = len(embs)
        if n >= 2:
            sim = cosine_similarity(np.array(embs))
            intra_scores.append((sim.sum() - n) / (n * (n - 1)))
    intra = float(np.mean(intra_scores)) if intra_scores else 1.0

    # --- inter : matrice chunk x chunk (chunk_embs déjà calculés) ---
    n = len(chunks)
    if n < 2:
        inter = 0.0
    else:
        sim_matrix = cosine_similarity(chunk_embs)
        inter = float((sim_matrix.sum() - n) / (n * (n - 1)))
    return intra, inter

def retrieval_metrics_full(chunk_embs, chunks, source_sentences, embed_fn, k=2):
    """Recall@k, Precision@k, F1, MRR sur des pseudo-questions (phrases du corpus)."""
    if len(chunks) < 2 or not source_sentences:
        return 0.0, 0.0, 0.0, 0.0
    queries = source_sentences
    q_embs = np.array(embed_fn(queries))
    hits, ranks, total = 0, [], 0
    for qi in range(len(queries)):
        sent = queries[qi]
        gt_idx = next((i for i, c in enumerate(chunks) if sent[:40] in c), None)
        if gt_idx is None:
            continue
        sims = cosine_similarity(q_embs[qi].reshape(1, -1), chunk_embs)[0]
        order = np.argsort(sims)[::-1]
        rank = int(np.where(order == gt_idx)[0][0]) + 1
        ranks.append(rank)
        total += 1
        if gt_idx in order[:k]:
            hits += 1
    if total == 0:
        return 0.0, 0.0, 0.0, 0.0
    recall = hits / total
    precision = hits / (total * k)
    f1 = 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    mrr = float(np.mean([1.0 / r for r in ranks]))
    return recall, precision, f1, mrr

source_sentences = simple_sentence_split(corpus_text)
print(f"{len(source_sentences)} phrases sources détectées (pseudo-questions).")


In [ ]:
# ---------- Boucle d'évaluation des 7 méthodes ----------
rows = []
method_chunk_embeddings = {}
for name, chunks in methods.items():
    print(f"Évaluation : {name} ({len(chunks)} chunks)...")
    chunk_embs = np.array(embed_fn(chunks))
    method_chunk_embeddings[name] = chunk_embs
    n, mean_len, std_len = length_stats(chunks)
    intra, inter = intra_inter_similarity(chunks, chunk_embs, embed_fn)
    recall, precision, f1, mrr = retrieval_metrics_full(chunk_embs, chunks, source_sentences[:150], embed_fn, k=2)
    rows.append({
        "Méthode": name, "Nb chunks": n, "Long. moyenne": round(mean_len, 1),
        "Écart-type": round(std_len, 1), "Similarité intra": round(intra, 3),
        "Similarité inter": round(inter, 3), "Recall@2": round(recall, 3),
        "Precision@2": round(precision, 3), "F1-score": round(f1, 3), "MRR": round(mrr, 3),
    })

results_df = pd.DataFrame(rows).set_index("Méthode")
results_df


## Meilleure méthode de chunking

**Score composite** = Similarité intra (cohérence) − Similarité inter (redondance) + F1-score.
Les **% d'écart** sont calculés par rapport à la médiane des 7 méthodes : ils montrent de **combien** chaque méthode se distingue, pas seulement qui gagne.


In [ ]:
def add_relative_percentages(df, metrics, lower_better=("Similarité inter",)):
    med = df[metrics].median()
    pct = pd.DataFrame(index=df.index)
    for m in metrics:
        sign = -1.0 if m in lower_better else 1.0
        denom = med[m] if med[m] != 0 else 1e-9
        pct[f"{m} %"] = ((sign * (df[m] - med[m])) / abs(denom) * 100).round(1)
    return pct

metrics_pct_metrics = ["Nb chunks", "Long. moyenne", "Similarité intra", "Similarité inter", "F1-score", "MRR"]
pct_df = add_relative_percentages(results_df, metrics_pct_metrics, lower_better=("Similarité inter",))
results_df_pct = pd.concat([results_df, pct_df], axis=1)

results_df["Score composite"] = (
    results_df["Similarité intra"] - results_df["Similarité inter"] + results_df["F1-score"]
).round(3)
ranking = results_df.sort_values("Score composite", ascending=False)
print(ranking[["Similarité intra", "Similarité inter", "F1-score", "MRR", "Score composite"]].to_string())
best_chunking_name = ranking.index[0]
print(f"\n>>> Meilleure méthode de chunking : {best_chunking_name}")
results_df_pct


In [ ]:
# Justification écrite de la meilleure méthode (chiffres à l'appui)
w = results_df.loc[best_chunking_name]
med = results_df[["Similarité intra", "Similarité inter", "F1-score", "MRR", "Nb chunks"]].median()
second = ranking.index[1]

chunking_justification = (
    f"La méthode « {best_chunking_name} » arrive en tête avec un score composite de {w['Score composite']:.3f} "
    f"(devant « {second} », {w['Score composite'] - ranking['Score composite'].iloc[1]:.3f} point plus haut). "
    f"Elle combine la meilleure cohérence interne ({w['Similarité intra']:.3f}, soit "
    f"{(w['Similarité intra']/med['Similarité intra']-1)*100:+.1f} % vs médiane), une redondance contenue "
    f"({w['Similarité inter']:.3f} contre {med['Similarité inter']:.3f} en médiane) et le meilleur comportement "
    f"en retrieval (F1 = {w['F1-score']:.3f}, MRR = {w['MRR']:.3f}) : ses chunks portent une idée homogène "
    f"sans dupliquer le contenu, ce qui maximise à la fois la précision et la position du bon passage dans le classement. "
    f"Avec {int(w['Nb chunks'])} chunks pour {int(med['Nb chunks'])} en médiane, le coût d'indexation reste raisonnable."
)
print(chunking_justification)


In [ ]:
# Visualisations : métriques clés +trade-off cohérence/redondance
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
colors = ["#f4a261" if m == best_chunking_name else "#4c78a8" for m in results_df.index]

results_df["F1-score"].plot(kind="barh", ax=axes[0], color=colors, title="F1-score (retrieval du bon chunk)")
axes[0].invert_yaxis()
results_df[["Similarité intra", "Similarité inter"]].plot(kind="barh", ax=axes[1],
               color=["#81b29a", "#e07a5f"], title="Cohérence intra (↑) vs redondance inter (↓)")
axes[1].invert_yaxis()
axes[2].scatter(results_df["Similarité inter"], results_df["Similarité intra"],
                s=results_df["F1-score"]*900, alpha=0.75, c=colors)
for m in results_df.index:
    axes[2].annotate(m.split(". ")[1], (results_df.loc[m, "Similarité inter"], results_df.loc[m, "Similarité intra"]),
                     fontsize=8, xytext=(4, 4), textcoords="offset points")
axes[2].set_xlabel("Redondance (similarité inter, ↓ mieux)")
axes[2].set_ylabel("Cohérence (similarité intra, ↑ mieux)")
axes[2].set_title("Trade-off cohérence / redondance (taille = F1)")
plt.tight_layout()
plt.savefig("docs/chunking_compare.png", dpi=130)
plt.show()


In [ ]:
# Export JSON de la comparaison de chunking (source de vérité de l'application web)
import json, os

os.makedirs("results", exist_ok=True)
chunking_export = {
    "stage": "chunking",
    "metrics_columns": ["Nb chunks", "Long. moyenne", "Écart-type", "Similarité intra",
                        "Similarité inter", "Recall@2", "Precision@2", "F1-score", "MRR", "Score composite"],
    "higher_is_better": {"Nb chunks": False, "Long. moyenne": None, "Écart-type": False,
                         "Similarité intra": True, "Similarité inter": False, "Recall@2": True,
                         "Precision@2": True, "F1-score": True, "MRR": True, "Score composite": True},
    "methods": [
        {"name": idx, **{col: (float(row[col]) if isinstance(row[col], (int, float, np.floating)) else row[col])
                          for col in results_df.columns}}
        for idx, row in results_df.iterrows()
    ],
    "best": {"name": best_chunking_name, "justification": chunking_justification},
    "ranking": list(ranking.index),
}
with open("results/chunking.json", "w", encoding="utf-8") as f:
    json.dump(chunking_export, f, ensure_ascii=False, indent=2, default=str)
print("results/chunking.json écrit.")

best_chunks = methods[best_chunking_name]
print(f"Chunks gagnants gardés en mémoire pour la suite : {len(best_chunks)} chunks.")


# PARTIE 3 — Comparaison des 7 méthodes d'Embedding

**Point de méthodologie** : le chunking est désormais **fixé** au gagnant de la Partie 2. On ne fait varier **que** la méthode de vectorisation.

## Les 7 méthodes

| # | Méthode | Principe | Entraînement | Points forts | Limites |
|---|---|---|---|---|---|
| 1 | **Word2Vec** | un vecteur par mot, agrégé par moyenne | sur le corpus local | léger, rapide, sans internet | n'a jamais vu le mot hors corpus ; perd l'ordre des mots |
| 2 | **GloVe** | co-occurrences globales, moyenne par chunk | pré-entraîné (Wikipedia) | vocabulaire mondial | moyenne de mots : le sens d'une phrase n'est pas un mot |
| 3 | **FastText** | mots + sous-mots (n-grammes de caractères) | sur le corpus local | gère les mots inconnus (OOV) via les sous-mots | même agrégation naïve que Word2Vec |
| 4 | **BERT (mean pooling)** | vecteurs **contextuels** (le même mot ≠ même vecteur selon le contexte) | pré-entraîné | capture le contexte local | brut, non optimisé pour la similarité de phrases |
| 5 | **Sentence-BERT (MiniLM)** | BERT re-entraîné pour que « phrases proches en sens = vecteurs proches » | pré-entraîné | LA méthode conçue pour le retrieval sémantique | — |
| 6 | **Doc2Vec** | un vecteur appris par document/chunk | sur le corpus local | vecteur de document natif | `infer_vector` non déterministe ; petit corpus |
| 7 | **Ollama (nomic-embed-text)** | modèle d'embedding moderne servi localement | pré-entraîné, local via API | pipeline 100 % local (sujet : Ollama) | nécessite Ollama actif |

**Hypothèse à vérifier** : les méthodes entraînées from scratch sur notre petit corpus (Word2Vec, FastText, Doc2Vec) devraient être nettement dominées par les pré-entraînées (GloVe, BERT, SBERT, nomic). Le comparatif sert justement à le démontrer chiffres à l'appui.


In [ ]:
import glob
pdf_paths = sorted(glob.glob("*.pdf"))
print(f"Corpus fixé : {len(best_chunks)} chunks ({best_chunking_name}) — {len(corpus_text)} caractères")
print(f"PDF source présent : {pdf_paths}")


In [ ]:
# ---------- 1. Word2Vec (entraîné sur le corpus) ----------
from gensim.models import Word2Vec

tokenized = [c.lower().split() for c in best_chunks]
w2v_model = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=1, epochs=50)

def embed_word2vec(texts):
    out = []
    for t in texts:
        words = [w for w in t.lower().split() if w in w2v_model.wv]
        out.append(np.mean([w2v_model.wv[w] for w in words], axis=0) if words else np.zeros(100))
    return out
print("Word2Vec entraîné (corpus).")


In [ ]:
# ---------- 2. GloVe (pré-entraîné, téléchargement ~130 Mo au 1er lancement) ----------
import gensim.downloader as gensim_api

glove_model = gensim_api.load("glove-wiki-gigaword-100")

def embed_glove(texts):
    out = []
    for t in texts:
        words = [w for w in t.lower().split() if w in glove_model]
        out.append(np.mean([glove_model[w] for w in words], axis=0) if words else np.zeros(100))
    return out
print("GloVe pré-entraîné chargé.")


In [ ]:
# ---------- 3. FastText (entraîné sur le corpus) ----------
from gensim.models import FastText

ft_model = FastText(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=1, epochs=50)

def embed_fasttext(texts):
    return [ft_model.wv.get_sentence_vector(t.lower().split()) for t in texts]
print("FastText entraîné (corpus).")


In [ ]:
# ---------- 4. BERT (mean pooling, pré-entraîné) ----------
import torch
from transformers import AutoTokenizer, AutoModel

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")
bert_model.eval()

def embed_bert(texts, batch_size=32):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = bert_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
            outputs = bert_model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1)
            mean_pooled = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
            out.extend(mean_pooled.numpy())
    return out
print("BERT chargé (mean pooling, par batch).")


In [ ]:
# ---------- 5. Sentence-BERT (MiniLM, pré-entraîné) ----------
from langchain_huggingface import HuggingFaceEmbeddings

sbert_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def embed_sbert(texts):
    return sbert_model.embed_documents(texts)
print("Sentence-BERT (MiniLM) chargé.")


In [ ]:
# ---------- 6. Doc2Vec (entraîné sur le corpus) ----------
from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument

tagged_docs = [TaggedDocument(words=tok, tags=[str(i)]) for i, tok in enumerate(tokenized)]
d2v_model = Doc2Vec(tagged_docs, vector_size=100, window=5, min_count=1, workers=1, epochs=50)

def embed_doc2vec(texts):
    return [d2v_model.infer_vector(t.lower().split(), epochs=30) for t in texts]
print("Doc2Vec entraîné (corpus).")


## Les 8 métriques

| Métrique | Objectif | Calcul |
|---|---|---|
| Similarité intra | **cohérence** thématique des chunks selon cet espace | cosinus moyen entre phrases d'un même chunk (↑ mieux) |
| Similarité inter | **séparation** entre chunks (↓ mieux) | cosinus moyen entre chunks différents |
| Precision@2 | pertinence des top-2 | hits / (2 × nb requêtes) |
| Recall@2 | couverture | le bon chunk est-il dans le top-2 ? |
| MAP | performance globale de classement | moyenne de 1/rang du bon chunk |
| MRR | position du bon résultat | identique à MAP ici : **une seule** réponse pertinente par requête → MAP = MRR (résultat théorique, pas une erreur) |
| nDCG | qualité du classement | 1/log₂(rang+1), pénalise un bon résultat trouvé tard |
| Cosine sim. (moy) | similarité brute question↔bon chunk | cosinus moyen sur les paires (question, bon chunk) |

**Même méthodologie que la Partie 2** : phrases = pseudo-questions, bon chunk = celui qui contient la phrase. Ici, la même famille de chunks est embeddée par **7 espaces vectoriels différents** : seuls les embeddings changent.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def intra_inter_for_embed(chunks, embed_fn):
    """intra : cosinus moyen entre phrases d'un même chunk (échantillon capé) ; inter : cosinus moyen entre chunks."""
    all_sents, owner = [], []
    for ci, c in enumerate(chunks):
        for s in simple_sentence_split(c):
            all_sents.append(s)
            owner.append(ci)
    if len(all_sents) > 2500:
        stride = int(np.ceil(len(all_sents) / 2500))
        all_sents, owner = all_sents[::stride], owner[::stride]
    sent_embs = np.array(embed_fn(all_sents)) if all_sents else np.zeros((0, 1))
    by_chunk = {}
    for emb, ci in zip(sent_embs, owner):
        by_chunk.setdefault(ci, []).append(emb)
    intra_scores = []
    for ci, embs in by_chunk.items():
        n = len(embs)
        if n >= 2:
            sim = cosine_similarity(np.array(embs))
            intra_scores.append((sim.sum() - n) / (n * (n - 1)))
    intra = float(np.mean(intra_scores)) if intra_scores else 1.0
    chunk_embs = np.array(embed_fn(chunks))
    n = len(chunks)
    if n < 2:
        inter = 0.0
    else:
        sim_matrix = cosine_similarity(chunk_embs)
        inter = float((sim_matrix.sum() - n) / (n * (n - 1)))
    return intra, inter, chunk_embs

def rank_metrics_for_embed(chunks, chunk_embs, embed_fn, source_sentences, k=2):
    """Precision@k, Recall@k, MAP, MRR, nDCG, cosinus moyen sur pseudo-questions."""
    if len(chunks) < 2 or not source_sentences:
        return dict(precision=0, recall=0, map_=0, mrr=0, ndcg=0, avg_cos=0)
    queries = source_sentences
    q_embs = np.array(embed_fn(queries))
    hits, ranks, cos_scores, total = 0, [], [], 0
    for qi in range(len(queries)):
        sent = queries[qi]
        gt_idx = next((i for i, c in enumerate(chunks) if sent[:40] in c), None)
        if gt_idx is None:
            continue
        sims = cosine_similarity(q_embs[qi].reshape(1, -1), chunk_embs)[0]
        order = np.argsort(sims)[::-1]
        rank = int(np.where(order == gt_idx)[0][0]) + 1
        ranks.append(rank)
        cos_scores.append(float(sims[gt_idx]))
        total += 1
        if gt_idx in order[:k]:
            hits += 1
    if total == 0:
        return dict(precision=0, recall=0, map_=0, mrr=0, ndcg=0, avg_cos=0)
    precision = hits / (total * k)
    recall = hits / total
    map_mrr = float(np.mean([1 / r for r in ranks]))
    ndcg = float(np.mean([1 / np.log2(r + 1) for r in ranks]))
    return dict(precision=round(precision, 3), recall=round(recall, 3), map_=round(map_mrr, 3),
                mrr=round(map_mrr, 3), ndcg=round(ndcg, 3), avg_cos=round(float(np.mean(cos_scores)), 3))

print("Fonctions de métriques embedding prêtes.")


In [ ]:
# ---------- 7. Ollama (nomic-embed-text, modèle local) ----------
import requests

_ollama_session = requests.Session()  # keep-alive pour accélérer les appels

def embed_ollama(texts, model="nomic-embed-text", url="http://localhost:11434/api/embed", batch_size=32):
    # Embeddings via l'API /api/embed (batchée) d'Ollama — rapide même sur des milliers de textes.
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        r = _ollama_session.post(url, json={"model": model, "input": batch}, timeout=120)
        r.raise_for_status()
        out.extend(r.json()["embeddings"])
    return out

try:
    _probe = embed_ollama(["test de disponibilité"])
    ollama_available = True
    print(f"Ollama disponible ({len(_probe[0])} dimensions).")
except Exception as e:
    ollama_available = False
    print(f"Ollama indisponible ({e}) -> la méthode sera notée N/A dans le tableau.")


In [ ]:
# ---------- Évaluation des 7 méthodes d'embedding ----------
embedding_methods = {
    "Word2Vec": embed_word2vec,
    "GloVe": embed_glove,
    "FastText": embed_fasttext,
    "BERT": embed_bert,
    "Sentence-BERT": embed_sbert,
    "Doc2Vec": embed_doc2vec,
}
if ollama_available:
    embedding_methods["Ollama"] = embed_ollama

emb_rows = []
embed_chunk_embs = {}
for name, fn in embedding_methods.items():
    print(f"Évaluation : {name}...")
    try:
        intra, inter, cembs = intra_inter_for_embed(best_chunks, fn)
        embed_chunk_embs[name] = cembs
        m = rank_metrics_for_embed(best_chunks, cembs, fn, source_sentences[:150], k=2)
        emb_rows.append({
            "Méthode": name, "Cosine sim. (moy)": m["avg_cos"], "Sim. intra": round(intra, 3),
            "Sim. inter": round(inter, 3), "Precision@2": m["precision"], "Recall@2": m["recall"],
            "MAP": m["map_"], "MRR": m["mrr"], "NDCG": m["ndcg"],
        })
    except Exception as e:
        print(f"  ! {name} indisponible : {type(e).__name__} {str(e)[:100]}")
        emb_rows.append({"Méthode": name, "Cosine sim. (moy)": None, "Sim. intra": None,
                          "Sim. inter": None, "Precision@2": None, "Recall@2": None,
                          "MAP": None, "MRR": None, "NDCG": None})

import pandas as pd
emb_results_df = pd.DataFrame(emb_rows).set_index("Méthode")
emb_results_df


## Meilleure méthode d'embedding

**Score composite** = Sim. intra − Sim. inter + MRR + NDCG (cohérence, séparation, classement). Les % d'écart sont relatifs à la médiane.


In [ ]:
valid = emb_results_df.dropna(subset=["MRR"]).copy()
valid["Score composite"] = (valid["Sim. intra"] - valid["Sim. inter"] + valid["MRR"] + valid["NDCG"]).round(3)
emb_ranking = valid.sort_values("Score composite", ascending=False)
print(emb_ranking[["Sim. intra", "Sim. inter", "MRR", "NDCG", "Score composite"]].to_string())
best_embedding_name = emb_ranking.index[0]
print(f"\n>>> Meilleure méthode d'embedding : {best_embedding_name}")

emb_pct_metrics = ["Cosine sim. (moy)", "Sim. intra", "Sim. inter", "Precision@2", "Recall@2", "MAP", "NDCG"]
med_e = valid[emb_pct_metrics].median()
emb_pct = pd.DataFrame(index=valid.index)
for m in emb_pct_metrics:
    sign = -1.0 if m == "Sim. inter" else 1.0
    denom = med_e[m] if med_e[m] != 0 else 1e-9
    emb_pct[f"{m} %"] = ((sign * (valid[m] - med_e[m])) / abs(denom) * 100).round(1)
emb_results_df_pct = pd.concat([valid, emb_pct], axis=1)
emb_results_df_pct


In [ ]:
# Justification écrite de la meilleure méthode d'embedding
w = emb_results_df_pct.loc[best_embedding_name]
med = emb_results_df_pct[["Sim. intra", "Sim. inter", "MRR", "NDCG"]].median()
second = emb_ranking.index[1]

embedding_justification = (
    f"« {best_embedding_name} » domine avec un score composite de {w['Score composite']:.3f} "
    f"(+{w['Score composite'] - emb_ranking['Score composite'].iloc[1]:.3f} devant « {second} »). "
    f"Ses pseudo-questions retrouvent leur chunk au rang moyen {1/w['MRR']:.1f} (MRR = {w['MRR']:.3f}, "
    f"{(w['MRR']/med['MRR']-1)*100:+.1f} % vs médiane) et son classement est le plus fiable "
    f"(nDCG = {w['NDCG']:.3f}, soit {(w['NDCG']/med['NDCG']-1)*100:+.1f} %). "
    f"Cela confirme l'hypothèse de départ : un espace vectoriel pré-entraîné sur de très grands corpus "
    f"généralise bien au-delà de notre thèse de 100 pages, là où les modèles entraînés from scratch "
    f"(Word2Vec, FastText, Doc2Vec) n'ont vu que ~{len(corpus_text.split()):,} mots."
)
print(embedding_justification)


In [ ]:
# Visualisation : heatmaps des similarités chunk x chunk par méthode + barres MRR
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ok_methods = [m for m in emb_ranking.index if m in embed_chunk_embs]
emb_colors = ["#7209b7" if m == best_embedding_name else "#4c78a8" for m in emb_ranking.index]
emb_ranking["MRR"].plot(kind="barh", ax=axes[0], color=emb_colors, title="MRR par méthode d'embedding")
axes[0].invert_yaxis()

mats = {}
for m in ok_methods[:4]:
    mats[m] = cosine_similarity(embed_chunk_embs[m][:60])
vmax = max(float(m.max()) for m in mats.values())
im = axes[1].imshow(np.mean([mats[m] for m in mats], axis=0), cmap="viridis", vmin=0, vmax=max(0.3, vmax))
axes[1].set_title("Similarité chunk×chunk (moyenne des 4 meilleures, 60 chunks)")
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.savefig("docs/embedding_compare.png", dpi=130)
plt.show()


In [ ]:
# Export JSON de la comparaison d'embedding
embedding_export = {
    "stage": "embedding",
    "metrics_columns": ["Cosine sim. (moy)", "Sim. intra", "Sim. inter", "Precision@2",
                        "Recall@2", "MAP", "MRR", "NDCG", "Score composite"],
    "higher_is_better": {"Cosine sim. (moy)": True, "Sim. intra": True, "Sim. inter": False,
                         "Precision@2": True, "Recall@2": True, "MAP": True, "MRR": True,
                         "NDCG": True, "Score composite": True},
    "methods": [
        {"name": idx, **{col: (float(row[col]) if row[col] is not None and not pd.isna(row[col]) else None)
                          for col in emb_results_df_pct.columns}}
        for idx, row in emb_results_df_pct.iterrows()
    ],
    "best": {"name": best_embedding_name, "justification": embedding_justification},
    "ranking": list(emb_ranking.index),
}
with open("results/embedding.json", "w", encoding="utf-8") as f:
    json.dump(embedding_export, f, ensure_ascii=False, indent=2, default=str)
print("results/embedding.json écrit.")


# PARTIE 4 — Comparaison des 6 méthodes de Retrieval

**Points de méthodologie** : chunking fixé (gagnant Partie 2), embedding fixé (gagnant Partie 3). On ne fait varier **que** la stratégie de recherche.

## Les 6 méthodes

| # | Méthode | Principe | Points forts | Limites |
|---|---|---|---|---|
| 1 | **Similarité brute** | comparaison **exhaustive** question × tous les chunks (cosinus exact) | résultat exact, zéro index à construire | temps de recherche O(N) par requête |
| 2 | **ANN rapide (FAISS)** | index approximatif (HNSW) : ne parcourt qu'une fraction des vecteurs | très rapide sur de gros corpus | léger risque de rater le vrai plus proche voisin |
| 3 | **Mots-clés (BM25)** | score lexical : rareté des termes + saturation + longueur | excellent sur les **termes exacts** (noms, sigles), aucune IA nécessaire | aveugle aux synonymes et à la reformulation |
| 4 | **Dense spécialisé (DPR)** | deux encodeurs (question / passage) entraînés **spécifiquement** pour le QA | aligne l'espace des questions sur celui des passages | nécessite un modèle spécifique, plus lourd |
| 5 | **Hybride (BM25 + dense, RRF)** | fusionne les classements lexical et sémantique par **Reciprocal Rank Fusion** | robuste : lexical + sens, sans accord de poids délicat | deux index à maintenir |
| 6 | **Re-ranking 2 étapes** | 1er étage rapide (récupère ~25 candidats) puis **cross-encoder** qui lit question+passage **ensemble** | la meilleure précision : le cross-encoder croise les mots finement | plus lent (1 inférence par candidat) |

## Métriques (même famille que les étapes précédentes)

| Métrique | Objectif | Calcul |
|---|---|---|
| Precision@5 | pertinence des 5 résultats | hits / (5 × nb requêtes) |
| Recall@5 | couverture | tous les passages pertinents sont-ils dans le top-5 ? |
| MRR | position du premier pertinent | moyenne de 1/rang |
| MAP@5 | précision moyenne sur tout le top-5 | moyenne des précisions aux rangs où un passage pertinent est retrouvé (≠ MRR ici : plusieurs passages pertinents par question) |
| nDCG@5 | qualité du classement | DCG idéal / DCG obtenu (log2) |
| Cosine sim. | proximité sémantique brute | cosinus moyen (embedding question, embedding du 1er passage retourné), mesuré dans le même espace pour les 6 méthodes |
| Latence (ms) | vitesse de réponse par requête | temps moyen mesuré (↓ mieux) |

**Jeu d'évaluation** : 20 questions réalistes sur la thèse (pollution azotée, zones humides, dénitrification...), chacune annotée par les **chunks pertinents** (chunk dont le contenu recouvre ≥ 2 mots significatifs de la question → score de pertinence gradué 0-3 pour le nDCG).


In [ ]:
# ---------- Fixer les données de la comparaison ----------
import time

# 1) Chunks : gagnant du chunking, embeddés avec le gagnant de l'embedding
retrieval_chunks = best_chunks
chunk_embs = np.array(embed_fn(retrieval_chunks))          # espace du gagnant embedding (MiniLM)
chunk_embs_unit = chunk_embs / np.linalg.norm(chunk_embs, axis=1, keepdims=True)
norms = np.linalg.norm(chunk_embs, axis=1)

# 2) Questions de test réalistes + tokens significatifs pour l'annotation automatique
test_queries = [
    "Comment les zones humides réduisent-elles la pollution par les nitrates ?",
    "Quel est le rôle des zones tampons dans le bassin versant ?",
    "Comment la dénitrification élimine-t-elle les nitrates du sol ?",
    "Quel impact de l'intensification de l'agriculture sur la qualité de l'eau ?",
    "Comment mesure-t-on les flux de nitrate dans les zones humides ?",
    "Quel est le rôle des roseaux dans l'épuration des eaux ?",
    "Comment les zones humides de fond de vallée piègent-elles les sédiments ?",
    "Qu'est-ce que la régulation des pollutions azotées diffuses ?",
    "Comment la végétation des berges influence-t-elle les flux d'azote ?",
    "Quels facteurs contrôlent la dénitrification dans les sols ?",
    "Quel est l'effet du drainage agricole sur les nappes ?",
    "Comment les prairies réduisent-elles le lessivage des nitrates ?",
    "Quel est le rôle des bactéries dans le cycle de l'azote ?",
    "Comment la topographie catena influence-t-elle les flux d'eau ?",
    "Qu'est-ce qu'une zone humide riparienne et son fonctionnement ?",
    "Comment la rotation des cultures influence la lixiviation de l'azote ?",
    "Quels outils pour cartographier les zones humides ?",
    "Comment la matière organique des sols favorise-t-elle la dénitrification ?",
    "Quel est l'impact des pratiques agricoles sur les émissions de N2O ?",
    "Comment les zones humides contribuent-elles à la qualité des eaux de surface ?",
]
import re as _re2
def sig_stems(q):
    # Stems = 6 premiers caractères des mots significatifs (approximation de stemming français :
    # neutralise les flexions comme réduisent/réduction, zones/zone, bactéries/bactéries).
    return sorted({w[:6] for w in _re2.findall(r"[a-zàâäéèêëîïôöùûüç]{4,}", q.lower())
                   if w not in {"comment", "quel", "quelles", "quels", "quelle", "est", "dans", "pour", "avec",
                                "pourquoi", "rôle", "role", "impact", "effet", "facteurs", "outils", "qu'est",
                                "influence", "contribuent", "favorise", "mesure", "quels"}})

def annotate_relevance(chunks, queries):
    # Pertinence graduée 0-3 = fraction des stems de la question présents dans le chunk
    # (méthode neutre : n'avantage ni les méthodes lexicales ni les denses).
    rel = {}
    for qi, q in enumerate(queries):
        stems = sig_stems(q)
        if not stems:
            rel[qi] = np.zeros(len(chunks), dtype=float)
            continue
        scores = []
        for c in chunks:
            cl = c.lower()
            h = sum(1 for s in stems if s in cl)
            frac = h / len(stems)
            scores.append(min(3.0, round(frac * 3, 1)))
        rel[qi] = np.array(scores, dtype=float)
    return rel

relevance = annotate_relevance(retrieval_chunks, test_queries)
print(f"{len(retrieval_chunks)} chunks | {len(test_queries)} questions annotées (pertinence graduée 0-3).")


In [ ]:
# ---------- Les 6 fonctions de retrieval ----------
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

BM25_TOKEN = re.compile(r"[a-zàâäéèêëîïôöùûüç]{2,}")

def bm25_tokenize(t):
    return [w for w in BM25_TOKEN.findall(t.lower()) if w not in
            {"les", "des", "une", "dans", "pour", "avec", "sont", "cette", "leur", "plus", "mais",
             "que", "qui", "par", "sur", "entre", "ainsi", "cela", "afin", "ors", "donc"}]

# 1) Similarité brute : cosinus exact sur tous les chunks
def retrieve_brute(question, k=5):
    q = np.array(embed_fn([question]))
    sims = cosine_similarity(q, chunk_embs)[0]
    idx = np.argsort(sims)[::-1][:k]
    return list(idx), time.perf_counter()

# 2) ANN rapide : FAISS HNSW
import faiss
dim = chunk_embs.shape[1]
hnsw = faiss.IndexHNSWFlat(dim, 32)
hnsw.add(chunk_embs.astype("float32"))

def retrieve_ann(question, k=5):
    q = np.array(embed_fn([question])).astype("float32")
    _, idx = hnsw.search(q, k)
    return list(idx[0]), time.perf_counter()

# 3) BM25 (mots-clés)
bm25 = BM25Okapi([bm25_tokenize(c) for c in retrieval_chunks])

def retrieve_bm25(question, k=5):
    scores = bm25.get_scores(bm25_tokenize(question))
    idx = np.argsort(scores)[::-1][:k]
    return list(idx), time.perf_counter()

# 4) Dense spécialisé (DPR) : encodeur question + encodeur contexte
from sentence_transformers import SentenceTransformer
dpr_model = SentenceTransformer("sentence-transformers/multi-qa-MiniLM-L6-cos-v1")
dpr_chunk_embs = dpr_model.encode(retrieval_chunks, batch_size=64, show_progress_bar=False, normalize_embeddings=True)

def retrieve_dpr(question, k=5):
    q = dpr_model.encode([question], normalize_embeddings=True)
    sims = (q @ dpr_chunk_embs.T)[0]
    idx = np.argsort(sims)[::-1][:k]
    return list(idx), time.perf_counter()

# 5) Hybride : RRF(BM25, dense MiniLM)
def retrieve_hybrid(question, k=5, dense_top=25):
    q = np.array(embed_fn([question]))
    sims = cosine_similarity(q, chunk_embs)[0]
    dense_idx = np.argsort(sims)[::-1][:dense_top]
    bm_idx = np.argsort(bm25.get_scores(bm25_tokenize(question)))[::-1][:dense_top]
    rrf = {}
    for rank, i in enumerate(dense_idx):
        rrf[int(i)] = rrf.get(int(i), 0.0) + 1.0 / (60 + rank + 1)
    for rank, i in enumerate(bm_idx):
        rrf[int(i)] = rrf.get(int(i), 0.0) + 1.0 / (60 + rank + 1)
    idx = sorted(rrf, key=rrf.get, reverse=True)[:k]
    return idx, time.perf_counter()

# 6) Re-ranking : 1er étage dense (25 candidats) + cross-encoder
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_rerank(question, k=5, candidate_top=25):
    q = np.array(embed_fn([question]))
    sims = cosine_similarity(q, chunk_embs)[0]
    cand = np.argsort(sims)[::-1][:candidate_top]
    pairs = [(question, retrieval_chunks[i][:512]) for i in cand]
    ce_scores = cross_encoder.predict(pairs, batch_size=32, show_progress_bar=False)
    order = np.argsort(ce_scores)[::-1]
    return [int(cand[j]) for j in order[:k]], time.perf_counter()

retrieval_methods = {
    "1. Similarité brute": retrieve_brute,
    "2. ANN rapide (FAISS)": retrieve_ann,
    "3. Mots-clés (BM25)": retrieve_bm25,
    "4. Dense spécialisé (DPR)": retrieve_dpr,
    "5. Hybride (BM25+dense)": retrieve_hybrid,
    "6. Re-ranking (2 étapes)": retrieve_rerank,
}
print("6 fonctions de retrieval définies.")


In [ ]:
# ---------- Boucle d'évaluation des 6 méthodes ----------
def precision_recall_mrr_ndcg(ranked_idx, rel_row, k=5):
    """rel_row : pertinence graduée 0-3 par chunk. Pertinent si score >= 2."""
    topk = list(ranked_idx)[:k]
    gains = np.array([rel_row[i] for i in topk], dtype=float)
    relevant_mask = gains >= 2
    hits = int(relevant_mask.sum())
    precision = hits / k
    recall = hits / max(1, int((rel_row >= 2).sum()))
    first_rel = next((r + 1 for r, g in enumerate(topk) if rel_row[g] >= 2), None)
    mrr = (1.0 / first_rel) if first_rel else 0.0
    dcg = sum((2**g - 1) / np.log2(pos + 2) for pos, g in enumerate(gains))
    ideal = np.sort(rel_row)[::-1][:k]
    idcg = sum((2**g - 1) / np.log2(pos + 2) for pos, g in enumerate(ideal))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    # Average Precision@5 (= MAP@5 par requête) : précision moyenne aux rangs où un
    # passage pertinent est retrouvé. Plusieurs passages pertinents par question
    # ici, donc MAP != MRR (contrairement à l'étape embedding, 1 seul pertinent).
    n_relevant = max(1, int((rel_row >= 2).sum()))
    cum_hits, ap_sum = 0, 0.0
    for pos, g in enumerate(topk):
        if rel_row[g] >= 2:
            cum_hits += 1
            ap_sum += cum_hits / (pos + 1)
    ap = ap_sum / min(k, n_relevant)
    return precision, recall, mrr, ndcg, ap

retrieval_rows = []
retrieval_latencies = {}
for name, fn in retrieval_methods.items():
    precs, recalls, mrrs, ndcgs, aps, lats, coss = [], [], [], [], [], [], []
    for qi, q in enumerate(test_queries):
        t0 = time.perf_counter()
        idx, _ = fn(q, k=5)
        dt = (time.perf_counter() - t0) * 1000
        p, r, m, n, ap = precision_recall_mrr_ndcg(idx, relevance[qi], k=5)
        # Cosine sim. : proximité (embedding question, embedding du 1er passage retourné),
        # mesurée dans le MÊME espace (gagnant embedding) pour les 6 méthodes -> comparable.
        q_emb = np.array(embed_fn([q]))
        top1_emb = np.array(embed_fn([retrieval_chunks[idx[0]]]))
        cos = float(cosine_similarity(q_emb, top1_emb)[0][0])
        precs.append(p); recalls.append(r); mrrs.append(m); ndcgs.append(n)
        aps.append(ap); lats.append(dt); coss.append(cos)
    retrieval_rows.append({
        "Méthode": name,
        "Precision@5": round(float(np.mean(precs)), 3),
        "Recall@5": round(float(np.mean(recalls)), 3),
        "MRR": round(float(np.mean(mrrs)), 3),
        "MAP@5": round(float(np.mean(aps)), 3),
        "NDCG@5": round(float(np.mean(ndcgs)), 3),
        "Cosine sim.": round(float(np.mean(coss)), 3),
        "Latence (ms)": round(float(np.mean(lats)), 1),
    })
    print(f"{name:28s} P@5={np.mean(precs):.3f} R@5={np.mean(recalls):.3f} MRR={np.mean(mrrs):.3f} MAP={np.mean(aps):.3f} NDCG={np.mean(ndcgs):.3f} cos={np.mean(coss):.3f} lat={np.mean(lats):.1f}ms")

ret_results_df = pd.DataFrame(retrieval_rows).set_index("Méthode")
ret_results_df


## Meilleure méthode de retrieval

**Score qualité** = moyenne normalisée (0-1) de Precision@5, Recall@5, MRR, MAP@5, nDCG@5 et Cosine sim. ; puis **Score global** = 0.7 × qualité − 0.3 × latence normalisée (le re-ranking peut être un peu plus lent : c'est le compromis qualité/vitesse qui décide).


In [ ]:
q_cols = ["Precision@5", "Recall@5", "MRR", "MAP@5", "NDCG@5", "Cosine sim."]
norm = (ret_results_df[q_cols] - ret_results_df[q_cols].min()) / (ret_results_df[q_cols].max() - ret_results_df[q_cols].min() + 1e-9)
ret_results_df["Score qualité"] = norm.mean(axis=1).round(3)
lat_norm = (ret_results_df["Latence (ms)"] - ret_results_df["Latence (ms)"].min()) / (ret_results_df["Latence (ms)"].max() - ret_results_df["Latence (ms)"].min() + 1e-9)
ret_results_df["Score global"] = (0.7 * ret_results_df["Score qualité"] - 0.3 * lat_norm).round(3)

ret_ranking = ret_results_df.sort_values("Score global", ascending=False)
best_retrieval_name = ret_ranking.index[0]
print(ret_ranking.to_string())
print(f"\n>>> Meilleure méthode de retrieval : {best_retrieval_name}")

# Pourcentages vs médiane
ret_pct_metrics = ["Precision@5", "Recall@5", "MRR", "MAP@5", "NDCG@5", "Cosine sim.", "Latence (ms)"]
med_r = ret_results_df[ret_pct_metrics].median()
ret_pct = pd.DataFrame(index=ret_results_df.index)
for m in ret_pct_metrics:
    sign = -1.0 if m == "Latence (ms)" else 1.0
    denom = med_r[m] if med_r[m] != 0 else 1e-9
    ret_pct[f"{m} %"] = ((sign * (ret_results_df[m] - med_r[m])) / abs(denom) * 100).round(1)
ret_results_df_pct = pd.concat([ret_results_df, ret_pct], axis=1)
ret_results_df_pct


In [ ]:
# Justification écrite de la meilleure méthode de retrieval
w = ret_results_df.loc[best_retrieval_name]
med = ret_results_df[q_cols].median()
second = ret_ranking.index[1]

retrieval_justification = (
    f"« {best_retrieval_name} » obtient le meilleur score global ({w['Score global']:.3f}) : "
    f"nDCG@5 = {w['NDCG@5']:.3f} ({(w['NDCG@5']/med['NDCG@5']-1)*100:+.1f} % vs médiane), "
    f"MAP@5 = {w['MAP@5']:.3f} ({(w['MAP@5']/med['MAP@5']-1)*100:+.1f} %) et MRR = {w['MRR']:.3f} ({(w['MRR']/med['MRR']-1)*100:+.1f} %), "
    f"avec une latence maîtrisée ({w['Latence (ms)']:.1f} ms/requête). "
    f"Devant « {second} » ({ret_ranking['Score global'].iloc[1]:.3f}), elle gagne parce que "
    + ("le cross-encoder lit chaque candidat **avec** la question et capte les correspondances fines que les bi-encodeurs manquent, "
       "ce qui remonte les passages vraiment pertinents au sommet du classement, tandis que le premier étage dense "
       "garde la recherche rapide."
       if "Re-ranking" in best_retrieval_name else
       "elle combine les termes exacts (BM25) et le sens (dense) : quand la question reformule le document, le dense compense ; "
       "quand elle emploie le vocabulaire exact de la thèse, le lexical confirme — la fusion RRF cumule les deux forces."
       if "Hybride" in best_retrieval_name else
       "les questions de test emploient souvent le vocabulaire exact du document (noms techniques comme « dénitrification », "
       "« zones humides »), et le score BM25 capte ces correspondances lexicales avec une précision remarquable pour un coût "
       "de calcul dérisoire (aucun modèle d'embedding requis) — les approches denses compensent quand la question reformule le "
       "texte, mais sur ce corpus technique le signal lexical domine, et l'écart de latence (2 ms contre 100 ms+) rend le "
       "compromis qualité/vitesse imbattable.")
)
print(retrieval_justification)


In [ ]:
# Visualisation : trade-off qualité vs latence + barres par métrique
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
colors = ["#7209b7" if m == best_retrieval_name else "#4c78a8" for m in ret_results_df.index]
axes[0].scatter(ret_results_df["Latence (ms)"], ret_results_df["Score qualité"], s=160, c=colors, alpha=0.85)
for m in ret_results_df.index:
    axes[0].annotate(m.split(". ")[1], (ret_results_df.loc[m, "Latence (ms)"], ret_results_df.loc[m, "Score qualité"]),
                     fontsize=8, xytext=(5, 4), textcoords="offset points")
axes[0].set_xlabel("Latence (ms) ↓ mieux")
axes[0].set_ylabel("Score qualité (0-1)")
axes[0].set_title("Trade-off qualité / vitesse (violet = gagnant)")

ret_results_df[q_cols].plot(kind="barh", ax=axes[1], title="Précision / Rappel / MRR / nDCG@5")
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig("docs/retrieval_compare.png", dpi=130)
plt.show()


In [ ]:
# Export JSON de la comparaison de retrieval
retrieval_export = {
    "stage": "retrieval",
    "metrics_columns": ["Precision@5", "Recall@5", "MRR", "MAP@5", "NDCG@5", "Cosine sim.", "Latence (ms)", "Score qualité", "Score global"],
    "higher_is_better": {"Precision@5": True, "Recall@5": True, "MRR": True, "MAP@5": True, "NDCG@5": True,
                         "Cosine sim.": True, "Latence (ms)": False, "Score qualité": True, "Score global": True},
    "methods": [
        {"name": idx, **{col: float(row[col]) for col in ret_results_df.columns}}
        for idx, row in ret_results_df.iterrows()
    ],
    "best": {"name": best_retrieval_name, "justification": retrieval_justification},
    "ranking": list(ret_ranking.index),
}
with open("results/retrieval.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_export, f, ensure_ascii=False, indent=2, default=str)
print("results/retrieval.json écrit.")


In [ ]:
# ---------- best_methods.json : gagnant + justification pour chaque étape ----------
best_methods_export = {
    "chunking": {"name": best_chunking_name, "justification": chunking_justification,
                 "score": float(results_df.loc[best_chunking_name, "Score composite"])},
    "embedding": {"name": best_embedding_name, "justification": embedding_justification,
                  "score": float(emb_results_df_pct.loc[best_embedding_name, "Score composite"])},
    "retrieval": {"name": best_retrieval_name, "justification": retrieval_justification,
                  "score": float(ret_results_df.loc[best_retrieval_name, "Score global"])},
    "pipeline_final": {
        "chunking": best_chunking_name,
        "embedding": best_embedding_name,
        "retrieval": best_retrieval_name,
        "resume": ("Le pipeline retenu enchaîne : découpage « %s » (chunks cohérents, peu redondants), "
                   "vectorisation « %s » (espace sémantique pré-entraîné), recherche « %s » (meilleur compromis "
                   "qualité/latence sur nos 20 questions de test)."
                   % (best_chunking_name, best_embedding_name, best_retrieval_name)),
    },
}
with open("results/best_methods.json", "w", encoding="utf-8") as f:
    json.dump(best_methods_export, f, ensure_ascii=False, indent=2)
print("results/best_methods.json écrit.")
print(best_methods_export["pipeline_final"]["resume"])


# PARTIE 5 — Génération (traitement des requêtes) : Ollama + repli Hugging Face

Dernière étape du pipeline : les passages récupérés + la question sont envoyés à un **LLM** qui rédige la réponse en s'appuyant **uniquement** sur ces extraits (avec sources). Deux backends testés :
- **Ollama** (local, demandé par le sujet) : `llama3.2:1b` via l'API `http://localhost:11434/api/chat` ;
- **Hugging Face** (repli) : petit modèle de génération.

**Nouveauté (extra)** : un second mini-benchmark comparant 3 tailles de modèles Ollama (1B / 3B / 8B) sur 5 questions : latence par token et score de citation des sources — à présenter devant le jury.


In [ ]:
# ---------- Détection du backend LLM disponible ----------
import requests, json

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL = "llama3.2:1b"

def ollama_ok():
    try:
        models = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json().get("models", [])
        return any(m["name"].startswith("llama3.2") for m in models)
    except Exception:
        return False

ollama_ready = ollama_ok()
print("Ollama prêt :", ollama_ready)


In [ ]:
# ---------- Génération : Ollama (prioritaire) puis repli Hugging Face ----------
def generate_ollama(prompt, model=GEN_MODEL, max_tokens=300, temperature=0.2):
    r = requests.post(f"{OLLAMA_URL}/api/generate", json={
        "model": model, "prompt": prompt, "stream": False,
        "options": {"num_predict": max_tokens, "temperature": temperature},
    }, timeout=180)
    r.raise_for_status()
    return r.json()["response"]

_hf_pipeline = None
def generate_hf(prompt, max_new_tokens=250):
    global _hf_pipeline
    if _hf_pipeline is None:
        from transformers import pipeline
        _hf_pipeline = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct",
                                torch_dtype="auto", device_map="cpu")
    messages = [{"role": "user", "content": prompt}]
    out = _hf_pipeline(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

def generate(prompt):
    if ollama_ready:
        try:
            return generate_ollama(prompt), "ollama"
        except Exception as e:
            print(f"Ollama a échoué ({str(e)[:80]}) -> repli Hugging Face.")
    return generate_hf(prompt), "huggingface"

def build_prompt(question, passages):
    context = "\n\n".join(f"[Source {i+1}] {p[:600]}" for i, p in enumerate(passages))
    return (
        "Tu es un assistant documentaire. Réponds en français, de façon factuelle et concise (5 phrases maximum), "
        "UNIQUEMENT à partir des extraits ci-dessous. Cite les sources utilisées sous la forme [Source N]. "
        "Si les extraits ne permettent pas de répondre, dis-le clairement.\n\n"
        f"### Extraits du corpus\n{context}\n\n### Question\n{question}\n\n### Réponse :"
    )

print("Fonctions de génération prêtes (Ollama + repli HF).")


In [ ]:
# ---------- Démo end-to-end : question -> retrieval (méthode gagnante) -> réponse citée ----------
demo_questions = [
    "Comment les zones humides réduisent-elles la pollution par les nitrates ?",
    "Quels facteurs contrôlent la dénitrification dans les sols ?",
]
best_retrieval_fn = retrieval_methods[best_retrieval_name]

for q in demo_questions:
    idx, _ = best_retrieval_fn(q, k=3)
    passages = [retrieval_chunks[i] for i in idx]
    prompt = build_prompt(q, passages)
    answer, backend = generate(prompt)
    print("=" * 100)
    print(f"QUESTION : {q}")
    print(f"RETRIEVAL ({best_retrieval_name}) -> chunks {list(idx)} | GÉNÉRATION : {backend}")
    print("-" * 100)
    print(answer.strip()[:1200])
    print(f"\n[SOURCES] " + " | ".join(f"[{i+1}] {passages[i][:90]}..." for i in range(len(passages))))


In [ ]:
# ---------- Mini-benchmark LLM : 3 tailles de modèles (1B / 3B / 8B) sur 5 questions ----------
import time as _time

def bench_model(model_name, questions, retrieval_fn, chunks, k=3, max_tokens=200):
    tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
    names = [m["name"] for m in tags.get("models", [])]
    if not any(n.startswith(model_name.split(":")[0]) for n in names):
        return None
    rows = []
    for q in questions:
        idx, _ = retrieval_fn(q, k=k)
        passages = [chunks[i] for i in idx]
        prompt = build_prompt(q, passages)
        t0 = _time.perf_counter()
        try:
            resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
                "model": model_name, "prompt": prompt, "stream": False,
                "options": {"num_predict": max_tokens, "temperature": 0.2}}, timeout=300)
            resp.raise_for_status()
            answer = resp.json()["response"]
            dt = (_time.perf_counter() - t0)
            n_tok = resp.json().get("eval_count", max(1, len(answer.split())))
            toks = [w for w in q.lower().split() if len(w) > 5]
            cited = sum(1 for t in toks if t in answer.lower())
            rows.append({"latence_s": round(dt, 2), "tokens": int(n_tok),
                         "ms/token": round(dt * 1000 / max(1, n_tok), 1), "citations_ok": cited})
        except Exception:
            rows.append({"latence_s": None, "tokens": None, "ms/token": None, "citations_ok": 0})
    df = pd.DataFrame(rows)
    return {"model": model_name, "latence_moy_s": round(df["latence_s"].dropna().mean(), 2),
            "ms_par_token": round(df["ms/token"].dropna().mean(), 1), "citations_moy": round(df["citations_ok"].mean(), 1)}

bench_questions = test_queries[:5]
llm_rows = []
for model_name in ["llama3.2:1b", "llama3.2:3b", "llama3.1:8b"]:
    print(f"Benchmark {model_name} sur {len(bench_questions)} questions...")
    r = bench_model(model_name, bench_questions, best_retrieval_fn, retrieval_chunks)
    if r:
        llm_rows.append(r)
    else:
        print(f"  {model_name} non installé -> ignoré.")
llm_bench_df = pd.DataFrame(llm_rows)
llm_bench_df


In [ ]:
# Export JSON du benchmark LLM (les timeouts deviennent null, pas NaN)
import math as _math

def _clean_llm_rows(rows):
    return [{k: (None if isinstance(v, float) and _math.isnan(v) else v) for k, v in r.items()} for r in rows]

llm_rows = _clean_llm_rows(llm_rows)
with open("results/llm_benchmark.json", "w", encoding="utf-8") as f:
    json.dump({"models": llm_rows}, f, ensure_ascii=False, indent=2, allow_nan=False)
print("results/llm_benchmark.json écrit.")


# PARTIE 6 — Architecture du projet

Le notebook produit les **données** ; l'application web les **expose** :

```
┌────────────────────────────  OFFLINE (ce notebook)  ────────────────────────────┐
│ thèse_pdf.pdf ─► CLEANING ─► CORPUS 100 p. ─► CHUNKING ×7 ─► gagnant            │
│                                                    gagnant ─► EMBEDDING ×7 ─► gagnant │
│                                                                   gagnant ─► RETRIEVAL ×6 ─► gagnant │
│                                                                                │
│ Exports : results/*.json (comparaisons + gagnants justifiés)                    │
│           + artefacts (chunks gagnants, embeddings, index BM25, modèles)        │
└──────────────────────────────────────│─────────────────────────────────────────┘
                                        ▼
┌────────────────────────────  ONLINE (application web)  ─────────────────────────┐
│  FRONTEND React (Vite, Recharts)  ──REST──►  BACKEND FastAPI   ──►  Ollama      │
│  · vue RAG Workflow (question → réponse)        · GET /api/results/{stage}      │
│  · vues comparatives (graphiques animés)        · GET /api/best-methods         │
│  · bannière "meilleure méthode" + justification │ · POST /api/query (live+LLM)  │
│  · console de question                          · sert le build frontend        │
└─────────────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# ---------- Génération de l'image d'architecture (docs/architecture.png) ----------
# Pipeline LINÉAIRE : corpus -> chunking -> embedding -> retrieval (chaque gagnant nourrit l'étape suivante).
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11.5, 7))
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")

def box(x, y, w, h, label, color, fs=9):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                         linewidth=1.4, edgecolor="#333", facecolor=color))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=fs)

def arrow(x1, y1, x2, y2, style="-|>", color="#333", lw=1.3, ls="-"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle=style, color=color, lw=lw, linestyle=ls))

# ---- OFFLINE : flux linéaire d'évaluation ----
ax.text(0.3, 9.6, "OFFLINE — Notebook (source de vérité)", fontsize=11, fontweight="bold")

box(0.3, 8.5, 1.6, 0.9, "PDF brut\n(thèse 191 p.)", "#e07a5f")
box(2.3, 8.5, 1.8, 0.9, "CLEANING\n→ corpus 100 p.", "#f2cc8f")
box(4.5, 8.5, 1.8, 0.9, "CORPUS", "#f2cc8f")
arrow(1.9, 8.95, 2.3, 8.95)
arrow(4.1, 8.95, 4.5, 8.95)

box(1.2, 6.7, 2.0, 1.0, "CHUNKING\n7 méthodes → gagnant", "#81b29a")
box(3.7, 6.7, 2.0, 1.0, "EMBEDDING\n7 méthodes → gagnant", "#81b29a")
box(6.2, 6.7, 2.0, 1.0, "RETRIEVAL\n6 méthodes → gagnant", "#81b29a")
arrow(4.9, 8.5, 2.2, 7.7)   # corpus -> chunking
arrow(3.2, 7.2, 3.7, 7.2)   # chunking -> embedding (chunks gagnants)
arrow(5.7, 7.2, 6.2, 7.2)   # embedding -> retrieval (embeddings gagnants)

box(8.7, 6.7, 1.1, 1.0, "results/\n*.json", "#cb997e")
arrow(8.2, 7.2, 8.7, 7.2)   # retrieval -> exports JSON

# ---- ONLINE ----
ax.text(0.3, 5.1, "ONLINE — Application web (FastAPI + React)", fontsize=11, fontweight="bold")

box(0.3, 2.4, 3.4, 1.6, "FRONTEND React + Recharts\nvues comparatives, RAG Workflow,\nconsole de question", "#a8dadc")
box(4.6, 2.4, 3.3, 1.6, "BACKEND FastAPI\n/api/results · /api/best-methods\n/api/query (live + LLM)", "#89b0ae")
box(8.5, 2.4, 1.2, 1.6, "Ollama\n(llama3.2,\nnomic-embed)", "#bee9e8")
arrow(3.7, 3.2, 4.6, 3.2, style="<->")
arrow(7.9, 3.2, 8.5, 3.2, style="<->")

box(4.6, 0.6, 3.3, 0.9, "Notebook → artefacts (chunks, modèles)", "#ffe8d6")
arrow(6.25, 1.5, 6.25, 2.4)              # artefacts -> backend (moteur live)
arrow(9.2, 6.7, 7.0, 4.0, ls=":", color="#888")  # exports JSON -> backend (vues comparatives)

plt.tight_layout()
plt.savefig("docs/architecture.png", dpi=140, bbox_inches="tight")
plt.show()
print("docs/architecture.png généré.")


# PARTIE 7 — Artefacts pour l'application web

Le backend FastAPI ne recalcule **rien** : il recharge les artefacts exportés ici (chunks gagnants, embeddings gagnants, index BM25/FAISS, modèles) pour offrir la console de question en live, et lit les JSON de comparaison pour les vues graphiques.


In [ ]:
# ---------- Export des artefacts du pipeline gagnant ----------
import pickle

artifacts = {
    "corpus_path": "data/CORPUS_RAG_reduit_100pages.txt",
    "corpus_chars": len(corpus_text),
    "cleaning": {k: v for k, v in info.items() if k != "text"},
    "winners": {
        "chunking": best_chunking_name,
        "embedding": best_embedding_name,
        "retrieval": best_retrieval_name,
    },
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "dpr_model": "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    "cross_encoder_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "n_chunks": len(best_chunks),
    "embedding_dim": int(chunk_embs.shape[1]),
    "llm": {"ollama_ready": bool(ollama_ready), "model": GEN_MODEL,
            "benchmark": llm_rows},
    "test_queries": test_queries,
}
with open("results/artifacts.json", "w", encoding="utf-8") as f:
    json.dump(artifacts, f, ensure_ascii=False, indent=2)

with open("results/chunks.json", "w", encoding="utf-8") as f:
    json.dump({"chunking_method": best_chunking_name, "chunks": best_chunks},
              f, ensure_ascii=False)

# Embeddings + BM25 + DPR : pickles (rechargés par le backend)
import faiss as _faiss
with open("results/embeddings_gagnant.pkl", "wb") as f:
    pickle.dump({"chunks": best_chunks,
                 "chunk_embeddings": chunk_embs,
                 "dpr_chunk_embs": dpr_chunk_embs,
                 "bm25": bm25,
                 "embedding_model": artifacts["embedding_model"],
                 "dpr_model": artifacts["dpr_model"],
                 "cross_encoder_model": artifacts["cross_encoder_model"]}, f)
print("results/artifacts.json, chunks.json, embeddings_gagnant.pkl écrits.")


# Synthèse finale

| Étape | Méthode gagnante | Pourquoi (résumé) |
|---|---|---|
| **Cleaning** | nettoyage automatique par détection de sections | 191 → 100 pages, retraits exacts demandés par l'encadrante |
| **Chunking** | voir `results/best_methods.json` | meilleur couple cohérence interne / redondance / F1 |
| **Embedding** | voir `results/best_methods.json` | espace pré-entraîné : meilleur MRR/nDCG sur les pseudo-questions |
| **Retrieval** | voir `results/best_methods.json` | meilleur compromis qualité de classement / latence |
| **Génération** | Ollama (repli HF) | réponses citées, 100 % local, latence mesurée par modèle |

**Livrables du projet** :
1. ce notebook exécuté de bout en bout (nettoyage → 3 comparaisons → génération) ;
2. `results/*.json` + `docs/*.png` pour le mémoire ;
3. l'application web FastAPI + React qui présente ces comparaisons avec la meilleure méthode **désignée et justifiée** à chaque étape, et la console de question branchée sur le pipeline gagnant.


In [ ]:
print("=== RÉCAPITULATIF DES EXPORTS ===")
import os
for f in sorted(os.listdir("results")):
    size = os.path.getsize(os.path.join("results", f))
    print(f"  results/{f:28s} {size/1024:8.1f} Ko")
for f in sorted(os.listdir("docs")):
    size = os.path.getsize(os.path.join("docs", f))
    print(f"  docs/{f:32s} {size/1024:8.1f} Ko")
print("\nNotebook terminé — l'application web peut être lancée : uvicorn backend.main:app --port 8001")
